# Script 25 — Tabelas suplementares do artigo
## Versão 4 — interseção vetorial otimizada

Esta versão elimina o processamento dos rasters anuais de soja para identificar MRS, REC e biomas.

**Lógica da TS01**
- Área anual de soja: base oficial IBGE/SIDRA.
- MRS, REC e biomas interceptados: interseção vetorial direta com as UFs.

Isso evita o gargalo que causou timeout nas versões anteriores.

In [ ]:
# =============================================================================
# CÉLULA 1 — BIBLIOTECAS E INICIALIZAÇÃO
# =============================================================================

import ee
import re

EE_PROJECT = "ee-marcosguedeslana"

try:
    ee.Initialize(project=EE_PROJECT)
    print("Earth Engine inicializado.")
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
    print("Earth Engine autenticado e inicializado.")

In [ ]:
# =============================================================================
# CÉLULA 2 — ASSETS E CONFIGURAÇÕES
# =============================================================================

ASSET_MICRO_BASE = (
    "projects/ee-marcosguedeslana/assets/Regiao_Edafoclimaticas/Microregiao"
)

ASSET_MACRO_BASE = (
    "projects/ee-marcosguedeslana/assets/Regiao_Edafoclimaticas/Macroregiao"
)

ASSET_ESTADOS = "projects/ee-marcosguedeslana/assets/IBGE/UF_2024"
ASSET_BIOMAS = "projects/ee-marcosguedeslana/assets/IBGE/Biomas"

CAMPO_ESTADO = "NM_UF"
CAMPO_BIOMA = "nm_bm"

MICROS = (
    list(range(101, 105)) +
    list(range(201, 205)) +
    list(range(301, 305)) +
    list(range(401, 406)) +
    list(range(501, 504))
)

MACROS = list(range(1, 6))

AREA_MIN_INTERSECAO_HA = 1
PASTA_DRIVE = "25_TABELAS_SUPLEMENTARES"

print("MICROS:", MICROS)
print("MACROS:", MACROS)

In [ ]:
# =============================================================================
# CÉLULA 3 — UFs DO ESTUDO
# =============================================================================

UFS = {
    "Mato Grosso": "MT",
    "Rio Grande do Sul": "RS",
    "Paraná": "PR",
    "Goiás": "GO",
    "Mato Grosso do Sul": "MS",
    "Minas Gerais": "MG",
    "Bahia": "BA",
    "São Paulo": "SP",
    "Tocantins": "TO",
    "Maranhão": "MA",
    "Piauí": "PI",
    "Santa Catarina": "SC",
    "Pará": "PA",
    "Rondônia": "RO",
    "Distrito Federal": "DF",
    "Roraima": "RR",
    "Amapá": "AP",
}

print("Número de UFs:", len(UFS))

In [ ]:
# =============================================================================
# CÉLULA 4 — FUNÇÕES PARA IDENTIFICAR MRS E REC
# =============================================================================

def normalizar_codigo(valor):
    if valor is None or isinstance(valor, bool):
        return None

    if isinstance(valor, (int, float)):
        try:
            return int(valor)
        except Exception:
            return None

    texto = str(valor).strip()

    try:
        return int(float(texto))
    except Exception:
        pass

    achados = re.findall(r"(\d+)", texto)
    return int(achados[-1]) if achados else None


def detectar_campo_codigo(fc, codigos_esperados, rotulo):
    esperados = set(int(x) for x in codigos_esperados)

    primeiro = ee.Feature(fc.first())
    propriedades = primeiro.propertyNames().getInfo()

    print(f"\n{rotulo} — propriedades:")
    print(propriedades)

    candidatos = []

    for prop in propriedades:
        try:
            valores = fc.aggregate_array(prop).distinct().getInfo()
        except Exception:
            continue

        normalizados = {
            normalizar_codigo(v)
            for v in valores
            if normalizar_codigo(v) is not None
        }

        if esperados.issubset(normalizados):
            candidatos.append((prop, valores, normalizados))

    if not candidatos:
        print("Primeira feição:")
        print(primeiro.toDictionary().getInfo())
        raise ValueError(
            f"Não foi possível detectar o campo de código de {rotulo}."
        )

    candidatos.sort(
        key=lambda x: (
            len(x[2] - esperados),
            len(x[2]),
            x[0]
        )
    )

    campo, valores, _ = candidatos[0]

    mapa = {
        codigo: [
            v for v in valores
            if normalizar_codigo(v) == codigo
        ]
        for codigo in sorted(esperados)
    }

    print(f"Campo selecionado para {rotulo}: {campo}")
    return campo, mapa


def montar_fc_codificada(base, codigos_esperados, rotulo):
    info = ee.data.getAsset(base)
    tipo = info.get("type")
    codigos_esperados = [int(x) for x in codigos_esperados]

    print("\n" + "=" * 70)
    print(rotulo)
    print("Asset:", base)
    print("Tipo:", tipo)
    print("=" * 70)

    if tipo == "TABLE":
        fc = ee.FeatureCollection(base)
        campo, mapa = detectar_campo_codigo(fc, codigos_esperados, rotulo)

        partes = []
        for codigo in codigos_esperados:
            sub = (
                fc
                .filter(ee.Filter.inList(campo, ee.List(mapa[codigo])))
                .map(
                    lambda f, codigo=codigo:
                    ee.Feature(f).set("REG_CODE", codigo)
                )
            )
            partes.append(sub)

        return ee.FeatureCollection(partes).flatten()

    if tipo == "FOLDER":
        resposta = ee.data.listAssets({"parent": base})
        filhos = resposta.get("assets", [])
        mapa_assets = {}

        print("Assets filhos:")
        for asset in filhos:
            nome = asset.get("name", "")
            print(" ", nome)

            basename = nome.rstrip("/").split("/")[-1]
            codigo = normalizar_codigo(basename)

            if codigo in codigos_esperados:
                mapa_assets[codigo] = nome

        faltantes = sorted(set(codigos_esperados) - set(mapa_assets))

        if faltantes:
            raise ValueError(
                f"{rotulo}: não foram encontrados assets para {faltantes}."
            )

        partes = []
        for codigo in codigos_esperados:
            sub = (
                ee.FeatureCollection(mapa_assets[codigo])
                .map(
                    lambda f, codigo=codigo:
                    ee.Feature(f).set("REG_CODE", codigo)
                )
            )
            partes.append(sub)

        return ee.FeatureCollection(partes).flatten()

    raise ValueError(f"{rotulo}: tipo de asset não suportado: {tipo}")

In [ ]:
# =============================================================================
# CÉLULA 5 — CARREGAR CAMADAS VETORIAIS
# =============================================================================

estados = ee.FeatureCollection(ASSET_ESTADOS)
biomas = ee.FeatureCollection(ASSET_BIOMAS)

fc_mrs = montar_fc_codificada(
    ASSET_MACRO_BASE,
    MACROS,
    "Macrorregiões Sojícolas — MRS"
)

fc_rec = montar_fc_codificada(
    ASSET_MICRO_BASE,
    MICROS,
    "Regiões Edafoclimáticas — REC"
)

nomes_biomas = (
    biomas
    .aggregate_array(CAMPO_BIOMA)
    .distinct()
    .sort()
    .getInfo()
)

print("\nBiomas encontrados:")
for nome in nomes_biomas:
    print(" -", nome)

In [ ]:
# =============================================================================
# CÉLULA 6 — FUNÇÕES DE INTERSEÇÃO VETORIAL
# =============================================================================

AREA_MIN_M2 = AREA_MIN_INTERSECAO_HA * 10000


def codigos_interceptados(fc_regioes, geom_uf):
    candidatos = fc_regioes.filterBounds(geom_uf)

    def calcular_area_intersecao(f):
        f = ee.Feature(f)

        area = (
            f.geometry()
            .intersection(geom_uf, ee.ErrorMargin(10))
            .area(maxError=10)
        )

        return f.set("INT_AREA_M2", area)

    validos = (
        candidatos
        .map(calcular_area_intersecao)
        .filter(ee.Filter.gt("INT_AREA_M2", AREA_MIN_M2))
    )

    return (
        ee.List(validos.aggregate_array("REG_CODE"))
        .distinct()
        .sort()
    )


def biomas_interceptados(geom_uf):
    candidatos = biomas.filterBounds(geom_uf)

    def calcular_area_intersecao(f):
        f = ee.Feature(f)

        area = (
            f.geometry()
            .intersection(geom_uf, ee.ErrorMargin(10))
            .area(maxError=10)
        )

        return f.set("INT_AREA_M2", area)

    validos = (
        candidatos
        .map(calcular_area_intersecao)
        .filter(ee.Filter.gt("INT_AREA_M2", AREA_MIN_M2))
    )

    return (
        ee.List(validos.aggregate_array(CAMPO_BIOMA))
        .distinct()
        .sort()
    )

print("Funções de interseção preparadas.")

In [ ]:
# =============================================================================
# CÉLULA 7 — GERAR RESUMO ESPACIAL DA TS01
# =============================================================================

resumo_features = []

for nome_uf, sigla in UFS.items():

    estado = ee.Feature(
        estados
        .filter(ee.Filter.eq(CAMPO_ESTADO, nome_uf))
        .first()
    )

    geom_uf = estado.geometry()

    mrs_lista = codigos_interceptados(fc_mrs, geom_uf)
    rec_lista = codigos_interceptados(fc_rec, geom_uf)
    bio_lista = biomas_interceptados(geom_uf)

    janela = "Mar-Oct" if sigla in {"RR", "AP"} else "Sep-May"

    feat = ee.Feature(
        None,
        {
            "UF": nome_uf,
            "SIGLA": sigla,
            "MRS_n": mrs_lista.size(),
            "MRS_codigos": mrs_lista.join("; "),
            "REC_n": rec_lista.size(),
            "REC_codigos": rec_lista.join("; "),
            "Biomas_n": bio_lista.size(),
            "Biomas_nomes": bio_lista.join("; "),
            "Janela_temporal": janela,
            "Criterio_intersecao": "vetorial_UF",
            "Area_min_intersecao_ha": AREA_MIN_INTERSECAO_HA,
        }
    )

    resumo_features.append(feat)

resumo = ee.FeatureCollection(resumo_features)

print("Resumo espacial preparado.")

In [ ]:
# =============================================================================
# CÉLULA 8 — INSPEÇÃO LEVE
# =============================================================================

preview = resumo.getInfo()

print("Número de UFs:", len(preview["features"]))
print()

for feat in preview["features"]:
    p = feat["properties"]

    print(
        p["SIGLA"],
        "| MRS:", p["MRS_codigos"],
        "| REC:", p["REC_codigos"],
        "| Biomas:", p["Biomas_nomes"]
    )

In [ ]:
# =============================================================================
# CÉLULA 9 — EXPORTAR RESUMO ESPACIAL DA TS01
# =============================================================================

task = ee.batch.Export.table.toDrive(
    collection=resumo,
    description="ts01_spatial_intersections_original",
    folder=PASTA_DRIVE,
    fileNamePrefix="ts01_spatial_intersections_original",
    fileFormat="CSV",
    selectors=[
        "UF",
        "SIGLA",
        "MRS_n",
        "MRS_codigos",
        "REC_n",
        "REC_codigos",
        "Biomas_n",
        "Biomas_nomes",
        "Janela_temporal",
        "Criterio_intersecao",
        "Area_min_intersecao_ha",
    ]
)

task.start()

print("Task iniciada.")
print("ID:", task.id)
print("Arquivo esperado: ts01_spatial_intersections_original.csv")

In [ ]:
# =============================================================================
# CÉLULA 10 — STATUS DA TASK
# =============================================================================

status = task.status()

print("Estado:", status.get("state"))

if status.get("error_message"):
    print("Erro:", status.get("error_message"))

## Saída esperada

`ts01_spatial_intersections_original.csv`

Depois, a área anual de soja da base consolidada IBGE/SIDRA será usada para completar a TS01 e construir a TS02.